# ECG ResNet18

單通道 ECG，辨識 NORMAL / VPC / AF。病人先依 seed 分組，再建立片段；禁止跨集合病人重疊。


In [ ]:
# 1. Seed 與病人層級切分設定（最先執行）
SEED = 171
VAL_SIZE = 0.15
TEST_SIZE = 0.20
NORMAL_TEST_SIZE = 0.20
VPC_TEST_SIZE = 0.25
AF_TEST_SIZE = 0.33

import os, random
import numpy as np
import torch

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("patient/model seed:", SEED)
print("病人切分在 window 建立前完成；train/validation/test 病人不得重複。")


In [ ]:
# 2. 路徑與模型參數
from pathlib import Path
import subprocess

PYTHON = Path(r"C:\Users\USER\AppData\Local\Programs\Python\Python311\python.exe")
PROJECT = Path(r"C:\Users\USER\Desktop\PECO_v2")
DATA_ROOT = Path("D:/Dataset_train")
OUTPUT_ROOT = Path("D:/PECO_ECG_runs/notebook_1107")
RUN_NAME = "ecg_res18_diseasef1_notebook_seed171"
RUN_ROOT = OUTPUT_ROOT / RUN_NAME

MODEL = "res18"
FILM_STAGE = 4
VPC_RULE_SCALE = 2
AF_RULE_SCALE = 8
VPC_WEIGHT = 1.1
AF_WEIGHT = 1.25
LEARNING_RATE = 0.0003


In [ ]:
# 3. 最終分類決策（直接使用已確定的值）
DECISION_RULE = "fixed_validation_thresholds"
FINAL_THRESHOLDS = {"VPC": 0.175, "AF": 0.275}
print("decision rule:", DECISION_RULE)
print("final thresholds:", FINAL_THRESHOLDS)


## 前處理

ECG 重採樣為 360 Hz；每段 2 秒、720 點。NORMAL 不重疊，VPC/AF 使用 1 秒滑動。VPC 片段須含 VPC-like beat，AF 片段須符合 RR 不規則條件。輸入為 `[batch, 1, 720]`，另加入 `[batch, 2]` 的 VPC/AF 規則分數。


In [ ]:
# 4. 訓練：checkpoint 只依 validation 的 VPC/AF 平均 F1 選擇
cmd = [
    str(PYTHON), str(PROJECT / "train_ecg_arrhythmia_multiclass.py"),
    "--data-root", str(DATA_ROOT), "--output-dir", str(OUTPUT_ROOT),
    "--locations", "all", "--models", MODEL,
    "--target-sr", "360", "--window-sec", "2",
    "--normal-stride-sec", "2", "--positive-stride-sec", "1",
    "--filter-vpc-candidate-windows", "--filter-af-irregular-windows",
    "--use-rule-feature-late-fusion", "--rule-feature-fusion-mode", "film",
    "--film-stage", str(FILM_STAGE), "--rule-feature-scale", "1",
    "--vpc-rule-feature-scale", str(VPC_RULE_SCALE),
    "--af-rule-feature-scale", str(AF_RULE_SCALE),
    "--rule-feature-csv", "D:/PECO_ECG_runs/all_window_rule_features/ecg_all_window_rule_features.csv",
    "--val-size", str(VAL_SIZE), "--test-size", str(TEST_SIZE),
    "--normal-test-size", str(NORMAL_TEST_SIZE),
    "--vpc-test-size", str(VPC_TEST_SIZE), "--af-test-size", str(AF_TEST_SIZE),
    "--seed", str(SEED), "--epochs", "50", "--batch-size", "128",
    "--lr", str(LEARNING_RATE), "--weight-decay", "0.0001",
    "--max-class-weight", "20", "--class-weight-power", "0",
    "--normal-weight-multiplier", "1",
    "--vpc-weight-multiplier", str(VPC_WEIGHT),
    "--af-weight-multiplier", str(AF_WEIGHT), "--label-smoothing", "0.05",
    "--select-level", "segment", "--select-metric", "disease_f1",
    "--early-stop-patience", "12", "--weighted-sampler",
    "--sampler-mode", "patient_class", "--cache-waveforms",
    "--num-workers", "0", "--device", "cuda", "--amp",
    "--run-name", RUN_NAME, "--no-progress",
]
subprocess.run(cmd, check=True)


In [ ]:
# 5. 檢查病人沒有洩漏
import json
model_dir = RUN_ROOT / "all" / MODEL
audit = json.loads((model_dir / "patient_split_integrity.json").read_text(encoding="utf-8"))
assert not audit["leakage_detected"]
print("patient leakage:", audit["leakage_detected"])
print("effective patients/windows:", audit["filtered_window_counts"])
